RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, PyMuPDFLoader, UnstructuredExcelLoader, BSHTMLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter ## <-- Changed here
from pathlib import Path

/var/folders/5h/4v7118bd3gv845rsl265wdp00000gn/T/ipykernel_2640/2972414263.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, PyMuPDFLoader, UnstructuredExcelLoader, BSHTMLLoader
/Users/jayanagasaivusa/notebook1/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
print("Ingesting all files from directories...")

# 1. Load PDFs
pdf_loader = DirectoryLoader("../data/pdf", glob="**/*.pdf", loader_cls=PyMuPDFLoader)
pdf_docs = pdf_loader.load()

# 2. Load Excel files
excel_loader = DirectoryLoader("../data/excel", glob="**/*.xlsx", loader_cls=UnstructuredExcelLoader)
excel_docs = excel_loader.load()

# 3. Load HTML files
html_loader = DirectoryLoader("../data/html", glob="**/*.html", loader_cls=BSHTMLLoader)
html_docs = html_loader.load()

all_pdf_documents = pdf_docs + excel_docs + html_docs
print(f"Success! Total documents loaded: {len(all_pdf_documents)}")

Ingesting all files from directories...
Success! Total documents loaded: 26


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'OpenOffice.org 2.4', 'creator': 'Writer', 'creationdate': '2009-03-24T11:33:15-06:00', 'source': '../data/pdf/bitcoin.pdf', 'file_path': '../data/pdf/bitcoin.pdf', 'total_pages': 9, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20090324113315-06'00'", 'page': 0}, page_content="Bitcoin: A Peer-to-Peer Electronic Cash System\nSatoshi Nakamoto\nsatoshin@gmx.com\nwww.bitcoin.org\nAbstract.  A purely peer-to-peer version of electronic cash would allow online \npayments to be sent directly from one party to another without going through a \nfinancial institution.  Digital signatures provide part of the solution, but the main \nbenefits are lost if a trusted third party is still required to prevent double-spending. \nWe propose a solution to the double-spending problem using a peer-to-peer network. \nThe network timestamps transactions by hashing them into an ongoi

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 26 documents into 88 chunks

Example chunk:
Content: Bitcoin: A Peer-to-Peer Electronic Cash System
Satoshi Nakamoto
satoshin@gmx.com
www.bitcoin.org
Abstract.  A purely peer-to-peer version of electronic cash would allow online 
payments to be sent dir...
Metadata: {'producer': 'OpenOffice.org 2.4', 'creator': 'Writer', 'creationdate': '2009-03-24T11:33:15-06:00', 'source': '../data/pdf/bitcoin.pdf', 'file_path': '../data/pdf/bitcoin.pdf', 'total_pages': 9, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20090324113315-06'00'", 'page': 0}


[Document(metadata={'producer': 'OpenOffice.org 2.4', 'creator': 'Writer', 'creationdate': '2009-03-24T11:33:15-06:00', 'source': '../data/pdf/bitcoin.pdf', 'file_path': '../data/pdf/bitcoin.pdf', 'total_pages': 9, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20090324113315-06'00'", 'page': 0}, page_content='Bitcoin: A Peer-to-Peer Electronic Cash System\nSatoshi Nakamoto\nsatoshin@gmx.com\nwww.bitcoin.org\nAbstract.  A purely peer-to-peer version of electronic cash would allow online \npayments to be sent directly from one party to another without going through a \nfinancial institution.  Digital signatures provide part of the solution, but the main \nbenefits are lost if a trusted third party is still required to prevent double-spending. \nWe propose a solution to the double-spending problem using a peer-to-peer network. \nThe network timestamps transactions by hashing them into an ongoi

embedding And vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8463.38it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/5h/4v7118bd3gv845rsl265wdp00000gn/T/ipykernel_2640/2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 25576.54it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/5h/4v7118bd3gv845rsl265wdp00000gn/T/ipykernel_2640/2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


Vectorstore

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 380


In [10]:
chunks

[Document(metadata={'producer': 'OpenOffice.org 2.4', 'creator': 'Writer', 'creationdate': '2009-03-24T11:33:15-06:00', 'source': '../data/pdf/bitcoin.pdf', 'file_path': '../data/pdf/bitcoin.pdf', 'total_pages': 9, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20090324113315-06'00'", 'page': 0}, page_content='Bitcoin: A Peer-to-Peer Electronic Cash System\nSatoshi Nakamoto\nsatoshin@gmx.com\nwww.bitcoin.org\nAbstract.  A purely peer-to-peer version of electronic cash would allow online \npayments to be sent directly from one party to another without going through a \nfinancial institution.  Digital signatures provide part of the solution, but the main \nbenefits are lost if a trusted third party is still required to prevent double-spending. \nWe propose a solution to the double-spending problem using a peer-to-peer network. \nThe network timestamps transactions by hashing them into an ongoi

In [11]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 88 texts...


Batches: 100%|██████████| 3/3 [00:00<00:00,  4.44it/s]

Generated embeddings with shape: (88, 384)
Adding 88 documents to vector store...
Successfully added 88 documents to vector store
Total documents in collection: 468


Retriever Pipeline From VectorStore

In [12]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = -2.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [13]:
rag_retriever

In [14]:
rag_retriever.retrieve("can you tell student name")

Retrieving documents for query: 'can you tell student name'
Top K: 5, Score threshold: -2.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.45it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_37a1ea61_18',
  'content': 'Note :\nAny dispute with regards to admission, eligibility, fees, refund, etc., shall be within the legal jurisdiction of\nChennai only.\na.\nb.\nc.\nStudent’s Signature Parent’s Signature\nName : VUSA MADHUBALA Name : VUSA VENKATA NARASIMHA\nRAO\nDeclaration:\nI hereby agree and abide by the above rules and regulations of SRMJCCA 2026. I am also aware that the\noriginal documents must be submitted online and the same will be verified in person at the time of\nenrollment. I have read and understood the refund norms of the University and would abide by the same.\nd. If the balance tuition fees is not paid by 31-05-2026\nSRM Nagar, Kattankulathur - 603203, Chengalpattu Dist., Tamil Nadu\nPhone: 080 6908 7000 Email: admissions.india@srmist.edu.in\nPrinted Date : 17-05-2026 08:33:54 PM\nRID - 729491 - EU1',
  'metadata': {'creationdate': '2026-05-17T20:33:54+05:30',
   'source': '../data/pdf/report-2.pdf',
   'creator': 'JasperReports Library versio

In [15]:
rag_retriever.retrieve("Who wrote the Bitcoin whitepaper?")

Retrieving documents for query: 'Who wrote the Bitcoin whitepaper?'
Top K: 5, Score threshold: -2.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 50.67it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_8e6da929_26',
  'content': 'the usual framework of coins made from digital signatures, which provides strong control of  \nownership, but is incomplete without a way to prevent double-spending.  To solve this, we  \nproposed a peer-to-peer network using proof-of-work to record a public history of transactions  \nthat quickly becomes computationally impractical  for  an attacker to change if honest nodes  \ncontrol a majority of CPU power.  The network is robust in its unstructured simplicity.  Nodes  \nwork all at once with little coordination.  They do not need to be identified, since messages are  \nnot routed to any particular place and only need to be delivered on a best effort basis.  Nodes can  \nleave  and  rejoin  the  network  at  will,  accepting  the  proof-of-work  chain  as  proof  of  what  \nhappened while they were gone.  They vote with their CPU power, expressing their acceptance of  \nvalid blocks by working on extending them and rejecting invalid blocks 

In [16]:
rag_retriever.retrieve("What are the different product categories or items listed in the retail sales data?")

Retrieving documents for query: 'What are the different product categories or items listed in the retail sales data?'
Top K: 5, Score threshold: -2.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 53.21it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_5db1ca08_54',
  'content': 'Order ID Customer Name Product Category Quantity Unit Price Total Amount Discount (%) Region Store Order Date Payment Method ORD-000001 Daniel Ramirez Cable Kit Accessories 29 13.04 378.16 0 North Store-03 04/28/2025 Wire Transfer ORD-000002 Christopher Hernandez Headphones Accessories 10 157.92 1263.36 20 East Store-11 09/24/2024 Debit Card ORD-000003 Linda Young Headphones Accessories 46 195.83 7656.95 15 South Store-08 08/03/2025 Cash ORD-000004 David Ramirez Standing Desk Furniture 45 635.82 27181.31 5 North Store-05 05/23/2025 Debit Card ORD-000005 Karen Allen Standing Desk Furniture 48 798.68 34502.98 10 West Store-19 05/21/2026 Check ORD-000006 Anthony Anderson Mouse Peripherals 50 40.9 2045 0 West Store-20 05/15/2026 Cash ORD-000007 Charles Ramirez USB Hub Accessories 37 24.32 854.85 5 West Store-20 07/28/2024 Debit Card ORD-000008 Jessica Johnson Standing Desk Furniture 40 329 11186 15 East Store-11 03/17/2026 Credit Card ORD-000009 Sus

In [17]:
rag_retriever.retrieve("What are the different button styles and form elements described?")

Retrieving documents for query: 'What are the different button styles and form elements described?'
Top K: 5, Score threshold: -2.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.85it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_e620465e_67',
  'content': 'HTML5 Website by Html5webtemplates.co.uk\n\n\nHome\n\nLayouts\n\nLeft Sidebar\nRight Sidebar\nNo Sidebar\n\nSubmenu\n\nOption 1\nOption 2\nOption 3\nOption 4\n\n\n\n\nElements\n\n\n\n\n\n\nMagna Aliquam\nSed lorem ipsum sed dolor nullam adipiscing\n\nLearn More\n\n\n\n\n\n\n\nLorem ipsum dolor\nIpsum dolor tempus commodo turpis adipiscing adipiscing in tempor placerat\n\t\t\t\t\t\tsed amet accumsan enim lorem sem turpis ut. Massa amet erat accumsan curae\n\t\t\t\t\t\tblandit porttitor faucibus in nisl nisi volutpat massa mi non nascetur.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nNatoque phasellus\nIpsum dolor tempus commodo amet sed accumsan et adipiscing blandit porttitor sed faucibus.\n\n\n\n\nUltricies dolore\nIpsum dolor tempus commodo amet sed accumsan et adipiscing blandit porttitor sed faucibus.\n\n\n\n\nMagna lacinia\nIpsum dolor tempus commodo amet sed accumsan et adipiscing blandit porttitor sed faucibus.\n\n\n\n\nPraesent l

In [18]:
rag_retriever.retrieve("What is mentioned under Magna Feugiat on the sidebar page?", score_threshold=-2.0)

Retrieving documents for query: 'What is mentioned under Magna Feugiat on the sidebar page?'
Top K: 5, Score threshold: -2.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 177.94it/s]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_e620465e_67',
  'content': 'HTML5 Website by Html5webtemplates.co.uk\n\n\nHome\n\nLayouts\n\nLeft Sidebar\nRight Sidebar\nNo Sidebar\n\nSubmenu\n\nOption 1\nOption 2\nOption 3\nOption 4\n\n\n\n\nElements\n\n\n\n\n\n\nMagna Aliquam\nSed lorem ipsum sed dolor nullam adipiscing\n\nLearn More\n\n\n\n\n\n\n\nLorem ipsum dolor\nIpsum dolor tempus commodo turpis adipiscing adipiscing in tempor placerat\n\t\t\t\t\t\tsed amet accumsan enim lorem sem turpis ut. Massa amet erat accumsan curae\n\t\t\t\t\t\tblandit porttitor faucibus in nisl nisi volutpat massa mi non nascetur.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nNatoque phasellus\nIpsum dolor tempus commodo amet sed accumsan et adipiscing blandit porttitor sed faucibus.\n\n\n\n\nUltricies dolore\nIpsum dolor tempus commodo amet sed accumsan et adipiscing blandit porttitor sed faucibus.\n\n\n\n\nMagna lacinia\nIpsum dolor tempus commodo amet sed accumsan et adipiscing blandit porttitor sed faucibus.\n\n\n\n\nPraesent l

RAG Pipeline- VectorDB To LLM Output Generation

In [19]:
from langchain_ollama import OllamaLLM
llm = OllamaLLM(model="gemma4:12b")

print("Connecting to local Ollama server...")

# 1. Initialize your local model (No API keys needed!)
llm = OllamaLLM(model="gemma4:12b") 

# 2. Define the question you want to ask
user_question = "What is mentioned under Magna Feugiat on the sidebar page?"

# 3. Retrieve the context using the custom retriever we built
# Using the negative threshold to ensure no results are dropped
raw_results = rag_retriever.retrieve(user_question, score_threshold=-2.0)

# 4. Extract just the text from the raw results
context_text = ""
for doc in raw_results:
    context_text += doc['content'] + "\n\n"

# 5. Build the prompt for Gemma
prompt = f"""
You are an intelligent assistant. Answer the user's question using ONLY the context provided below.
If the answer is not in the context, say "I don't know based on the provided documents."

Context:
{context_text}

User Question:
{user_question}

Answer:
"""

# 6. Send the prompt to your local model!
print("Thinking...\n")
response = llm.invoke(prompt)

print("Final Answer:")
print(response)

Connecting to local Ollama server...
Retrieving documents for query: 'What is mentioned under Magna Feugiat on the sidebar page?'
Top K: 5, Score threshold: -2.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 162.68it/s]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
Thinking...

Final Answer:
Sed tristique purus vitae volutpat commodo suscipit amet sed nibh. Proin a ullamcorper sed blandit. Sed tristique purus vitae volutpat commodo suscipit ullamcorper commodo suscipit amet sed nibh. Proin a ullamcorper sed blandit..


This is function for asking the questions to ai
if u want to ask anything just type ask_ai("question")

In [20]:
def ask_ai(question):
    raw_results = rag_retriever.retrieve(question, score_threshold=-2.0)
    context_text = "\n".join([doc['content'] for doc in raw_results])
    
    prompt = f"""You are a helpful assistant. Use ONLY the provided context to answer the question. 
    If the answer is not in the context, say "I don't know."
    
    Context: {context_text}
    Question: {question}
    Answer:"""
    
    return llm.invoke(prompt)

# Now you can test instantly!
print(ask_ai("Based on the retail sales data, who got 20% discount and when?"))
print(ask_ai("What is the 2026 Q3 revenue forecast?"))

Retrieving documents for query: 'Based on the retail sales data, who got 20% discount and when?'
Top K: 5, Score threshold: -2.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.39s/it]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
Christopher Hernandez received a 20% discount on 09/24/2024.
Retrieving documents for query: 'What is the 2026 Q3 revenue forecast?'
Top K: 5, Score threshold: -2.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.79it/s]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
I don't know.
